# 1-3. 얼굴 crop 품질 필터 수집기

MediaPipe로 얼굴, 손, 포즈를 확인하면서 CNN 학습용 얼굴 crop 데이터셋을 수집합니다. `1`, `2`, `3` 키를 눌렀을 때 선택한 클래스 점수가 기준값 이상인 경우에만 저장 대기열에 넣습니다.

## 1. import

웹캠, MediaPipe, 이미지 저장, 메타데이터 기록에 필요한 라이브러리를 불러옵니다.

In [10]:
import csv
import time
from datetime import datetime
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_styles, drawing_utils

## 2. 설정

수집 기준은 이 셀에서 조정합니다. 처음에는 너무 빡빡하게 잡지 말고, 수집해 보면서 threshold를 올리는 방식이 좋습니다.

In [11]:
WINDOW_NAME = "Focus On Class - Capture Quality Gate"
CAMERA_INDEX = 0
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

SAVE_ROOT = Path("captured_images")
SAVE_DIRS = {
    "Attractive": SAVE_ROOT / "Attractive",
    "Drowsy": SAVE_ROOT / "Drowsy",
    "Inattentive": SAVE_ROOT / "Inattentive",
}
METADATA_PATH = SAVE_ROOT / "metadata.csv"

for save_dir in SAVE_DIRS.values():
    save_dir.mkdir(parents=True, exist_ok=True)

# 선택한 클래스 점수가 기준 이상일 때만 캡처합니다.
CAPTURE_THRESHOLD = 0.60
CROP_OUTPUT_SIZE = (224, 224)
MIN_FACE_AREA_RATIO = 0.015

BLUE_OVERLAY_ALPHA = 0.35
FACE_BOX_COLOR = (0, 255, 255)
TEXT_COLOR = (255, 255, 255)

FACE_MODEL_PATH = r"mp_model/face_landmarker.task"
HAND_MODEL_PATH = r"mp_model/hand_landmarker.task"
POSE_MODEL_PATH = r"mp_model/pose_landmarker_full.task"

BaseOptions = mp.tasks.BaseOptions
RunningMode = mp.tasks.vision.RunningMode
FaceLandmarker = mp.tasks.vision.FaceLandmarker
HandLandmarker = mp.tasks.vision.HandLandmarker
PoseLandmarker = mp.tasks.vision.PoseLandmarker

face_options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=FACE_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

pose_options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=True,
)

## 3. 기본 유틸 함수

MediaPipe 입력 변환, 얼굴 bbox 계산, 얼굴 crop 저장 이미지를 만드는 작은 함수들입니다.

In [12]:
def make_mp_image(frame_rgb: np.ndarray) -> mp.Image:
    """Convert an RGB frame to the MediaPipe image format."""
    return mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(frame_rgb),
    )


def clamp01(value: float) -> float:
    return float(np.clip(value, 0.0, 1.0))


def face_landmarks_to_bbox(face_landmarks, frame_width: int, frame_height: int):
    """Convert face landmarks to a padded bounding box."""
    xs = [int(lm.x * frame_width) for lm in face_landmarks]
    ys = [int(lm.y * frame_height) for lm in face_landmarks]

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    pad_x = int((x2 - x1) * 0.12)
    pad_y = int((y2 - y1) * 0.12)

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(frame_width - 1, x2 + pad_x)
    y2 = min(frame_height - 1, y2 + pad_y)

    return x1, y1, x2, y2


def crop_first_face_bgr(frame_bgr: np.ndarray, face_boxes, output_size=CROP_OUTPUT_SIZE):
    """Make a CNN-ready crop from the first detected face box."""
    if not face_boxes:
        return None

    x1, y1, x2, y2 = face_boxes[0]
    cropped = frame_bgr[y1:y2 + 1, x1:x2 + 1]

    if cropped.size == 0:
        return None

    cropped = cv2.resize(cropped, output_size, interpolation=cv2.INTER_AREA)
    return cropped.copy()

## 4. 클래스 점수 함수

완성된 분류 모델이 아니라, 좋은 데이터만 모으기 위한 간단한 품질 필터입니다. 점수는 MediaPipe 얼굴 랜드마크에서 눈 뜨임, 고개 숙임, 얼굴 위치, 얼굴 크기를 이용해 계산합니다.

In [13]:
def calculate_class_scores(face_result, face_boxes, frame_shape):
    """Calculate class scores and metadata features for the current frame."""
    scores = {"Attractive": 0.0, "Drowsy": 0.0, "Inattentive": 0.0}
    features = {
        "face_area": 0.0,
        "eye_open": 0.0,
        "head_down": 0.0,
        "off_center": 1.0,
        "face_yaw": 1.0,
        "head_tilt": 1.0,
        "frontal": 0.0,
    }

    face_landmarks_list = getattr(face_result, "face_landmarks", [])
    if not face_landmarks_list or not face_boxes:
        return scores, features

    height, width = frame_shape[:2]
    x1, y1, x2, y2 = face_boxes[0]
    box_w = max(1, x2 - x1)
    box_h = max(1, y2 - y1)

    face_area = (box_w * box_h) / max(1, width * height)
    face_size_score = clamp01(face_area / MIN_FACE_AREA_RATIO)

    face_center_x = (x1 + x2) / 2
    face_center_y = (y1 + y2) / 2
    center_dx = abs(face_center_x - width / 2) / (width / 2)
    center_dy = abs(face_center_y - height / 2) / (height / 2)
    off_center = clamp01((center_dx + center_dy) / 1.2)
    centered = 1.0 - off_center

    face_landmarks = face_landmarks_list[0]
    eye_open = eye_open_score(face_landmarks, width, height)
    eye_closed = 1.0 - eye_open

    nose_y = face_landmarks[1].y * height
    nose_ratio = (nose_y - y1) / max(1, box_h)
    head_down = clamp01((nose_ratio - 0.52) / 0.18)

    face_yaw = face_yaw_score(face_landmarks, width)
    head_tilt = head_tilt_score(face_landmarks, width, height)
    frontal = clamp01(1.0 - max(face_yaw, head_tilt))

    # Build a stronger inattentive evidence term and use it to suppress Attractive.
    posture_break = clamp01(max(off_center, face_yaw, head_tilt))
    inattentive_evidence = clamp01(
        0.50 * posture_break
        + 0.20 * off_center
        + 0.20 * face_yaw
        + 0.10 * head_down
    )

    quality = min(face_size_score, 1.0)
    attractive_base = clamp01(
        0.30 * quality
        + 0.30 * centered
        + 0.20 * eye_open
        + 0.20 * frontal
    )
    attractive_gate = clamp01(1.0 - 0.75 * inattentive_evidence)
    scores["Attractive"] = clamp01(attractive_base * attractive_gate)

    scores["Drowsy"] = clamp01(
        0.20 * quality
        + 0.50 * eye_closed
        + 0.25 * head_down
        + 0.05 * head_tilt
    )
    scores["Inattentive"] = clamp01(
        0.10 * quality
        + 0.85 * inattentive_evidence
        + 0.05 * (1.0 - frontal)
    )

    features.update({
        "face_area": face_area,
        "eye_open": eye_open,
        "head_down": head_down,
        "off_center": off_center,
        "face_yaw": face_yaw,
        "head_tilt": head_tilt,
        "frontal": frontal,
    })
    return scores, features

## 5. 화면 표시 함수

수집 중에 얼굴 bbox, 손/포즈 랜드마크, 클래스 점수를 화면에 표시합니다.

In [14]:
def draw_blue_segmentation_overlay(frame_rgb: np.ndarray, pose_result) -> np.ndarray:
    annotated_frame = frame_rgb.copy()

    if not getattr(pose_result, "segmentation_masks", None):
        return annotated_frame

    segmentation_mask = np.squeeze(pose_result.segmentation_masks[0].numpy_view())
    if segmentation_mask.shape[:2] != annotated_frame.shape[:2]:
        segmentation_mask = cv2.resize(
            segmentation_mask,
            (annotated_frame.shape[1], annotated_frame.shape[0]),
            interpolation=cv2.INTER_LINEAR,
        )

    blue_overlay = np.zeros_like(annotated_frame, dtype=np.uint8)
    blue_overlay[:, :, 2] = (segmentation_mask * 255).astype(np.uint8)
    return cv2.addWeighted(annotated_frame, 1.0, blue_overlay, BLUE_OVERLAY_ALPHA, 0)


def draw_face_landmarks(frame_rgb: np.ndarray, face_result):
    annotated_frame = frame_rgb.copy()
    face_boxes = []
    height, width = annotated_frame.shape[:2]

    for face_idx, face_landmarks in enumerate(getattr(face_result, "face_landmarks", [])):
        x1, y1, x2, y2 = face_landmarks_to_bbox(face_landmarks, width, height)
        face_boxes.append((x1, y1, x2, y2))

        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FACE_BOX_COLOR, 2)
        cv2.putText(annotated_frame, f"Face {face_idx + 1}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, TEXT_COLOR, 2, cv2.LINE_AA)
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style(),
        )

    return annotated_frame, face_boxes


def draw_hand_landmarks(frame_rgb: np.ndarray, hand_result) -> np.ndarray:
    annotated_frame = frame_rgb.copy()
    for hand_landmarks in getattr(hand_result, "hand_landmarks", []):
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=hand_landmarks,
            connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
            landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style(),
        )
    return annotated_frame


def draw_pose_landmarks(frame_rgb: np.ndarray, pose_result) -> np.ndarray:
    annotated_frame = frame_rgb.copy()
    for pose_landmarks in getattr(pose_result, "pose_landmarks", []):
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2),
        )
    return annotated_frame


def draw_status(frame_rgb: np.ndarray, fps: float, face_count: int, hand_count: int, scores, features) -> np.ndarray:
    annotated_frame = frame_rgb.copy()
    lines = [
        f"FPS: {fps:.1f}",
        f"Faces: {face_count}  Hands: {hand_count}",
        f"Threshold: {CAPTURE_THRESHOLD:.2f}",
        f"1 Attractive: {scores['Attractive']:.2f}",
        f"2 Drowsy: {scores['Drowsy']:.2f}",
        f"3 Inattentive: {scores['Inattentive']:.2f}",
        f"Yaw: {features['face_yaw']:.2f}  Tilt: {features['head_tilt']:.2f}",
        "s: Save   q: Quit",
    ]

    y = 30
    for line in lines:
        cv2.putText(annotated_frame, line, (20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, TEXT_COLOR, 2, cv2.LINE_AA)
        y += 24

    return annotated_frame

## 6. 저장 함수

이미지는 클래스 폴더에 저장하고, 같은 기준으로 계산된 점수와 feature는 `metadata.csv`에 남깁니다.

In [15]:
def append_metadata(rows):
    """Append saved-image metadata to CSV."""
    if not rows:
        return

    METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    file_exists = METADATA_PATH.exists()
    fieldnames = ["path", "label", "score", "face_area", "eye_open", "head_down", "off_center", "face_yaw", "head_tilt", "frontal", "timestamp"]

    with METADATA_PATH.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def save_pending_images(pending_images):
    """Save queued crops and write metadata."""
    save_token = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    saved_rows = []

    for label, items in pending_images.items():
        save_dir = SAVE_DIRS[label]
        save_dir.mkdir(parents=True, exist_ok=True)
        prefix = label.lower()

        for index, item in enumerate(items, start=1):
            image_path = save_dir / f"{prefix}_{save_token}_{index:03d}.jpg"
            saved = cv2.imwrite(str(image_path), item["image"])

            if not saved:
                print(f"{label} save failed: {image_path}")
                continue

            row = {"path": str(image_path), "label": label, "score": round(item["score"], 4), "timestamp": save_token}
            row.update({key: round(value, 4) for key, value in item["features"].items()})
            saved_rows.append(row)
            print(f"{label} saved: {image_path}")

    append_metadata(saved_rows)
    return len(saved_rows)

## 7. 저장 폴더 준비

메인 루프를 실행하기 전에 `captured_images`와 라벨별 폴더를 다시 만듭니다. 수집 중에 폴더를 지웠더라도 저장 실패를 줄일 수 있습니다.

In [16]:
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

for label, save_dir in SAVE_DIRS.items():
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"ready: {label} -> {save_dir}")

print(f"metadata path: {METADATA_PATH}")

ready: Attractive -> captured_images\Attractive
ready: Drowsy -> captured_images\Drowsy
ready: Inattentive -> captured_images\Inattentive
metadata path: captured_images\metadata.csv


## 8. 메인 실행 루프

실행 후 웹캠 창에서 `1`, `2`, `3`을 누르면 해당 클래스 점수가 `CAPTURE_THRESHOLD` 이상일 때만 얼굴 crop이 저장 대기열에 들어갑니다. `s`를 누르면 파일로 저장하고, `q`를 누르면 종료합니다.

In [17]:
cv2.destroyAllWindows()

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
    FaceLandmarker.create_from_options(face_options) as face_landmarker, \
    HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    camera_capture = cv2.VideoCapture(CAMERA_INDEX)
    camera_capture.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    camera_capture.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

    if not camera_capture.isOpened():
        raise RuntimeError("Camera open failed. Check CAMERA_INDEX or camera permission.")

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    start_time = time.perf_counter()
    last_frame_time = time.perf_counter()
    # s를 누르기 전까지 캡처한 crop을 클래스별로 모아둡니다.
    pending_images = {label: [] for label in SAVE_DIRS}
    key_to_label = {ord("1"): "Attractive", ord("2"): "Drowsy", ord("3"): "Inattentive"}

    try:
        while True:
            is_readable, frame_bgr = camera_capture.read()
            if not is_readable:
                print("Frame read failed.")
                break

            # 거울처럼 보이도록 좌우를 반전합니다.
            frame_bgr = cv2.flip(frame_bgr, 1)
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = make_mp_image(frame_rgb)
            timestamp_ms = int((time.perf_counter() - start_time) * 1000)

            pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
            face_result = face_landmarker.detect_for_video(mp_image, timestamp_ms)
            hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

            annotated_frame_rgb = draw_blue_segmentation_overlay(frame_rgb, pose_result)
            annotated_frame_rgb = draw_pose_landmarks(annotated_frame_rgb, pose_result)
            annotated_frame_rgb, face_boxes = draw_face_landmarks(annotated_frame_rgb, face_result)
            annotated_frame_rgb = draw_hand_landmarks(annotated_frame_rgb, hand_result)

            # MediaPipe 결과로 현재 프레임의 클래스 점수를 계산합니다.
            scores, features = calculate_class_scores(face_result, face_boxes, frame_rgb.shape)

            current_time = time.perf_counter()
            fps = 1.0 / max(1e-6, current_time - last_frame_time)
            last_frame_time = current_time

            hand_count = len(getattr(hand_result, "hand_landmarks", []))
            annotated_frame_rgb = draw_status(annotated_frame_rgb, fps, len(face_boxes), hand_count, scores, features)

            display_frame_bgr = cv2.cvtColor(annotated_frame_rgb, cv2.COLOR_RGB2BGR)
            cv2.imshow(WINDOW_NAME, display_frame_bgr)

            key = cv2.waitKey(1) & 0xFF

            if key in key_to_label:
                label = key_to_label[key]
                score = scores[label]
                crop_frame_bgr = crop_first_face_bgr(frame_bgr, face_boxes)

                if crop_frame_bgr is None:
                    print(f"[{label}] no face, skipped.")
                    continue

                # 점수가 기준보다 낮으면 애매한 샘플로 보고 저장하지 않습니다.
                if score < CAPTURE_THRESHOLD:
                    print(f"[{label}] score {score:.2f} < {CAPTURE_THRESHOLD:.2f}, skipped.")
                    continue

                pending_images[label].append({"image": crop_frame_bgr, "score": score, "features": features.copy()})
                print(f"[{label}] score {score:.2f}, pending {len(pending_images[label])} images.")
                continue

            if key == ord("s"):
                saved_count = save_pending_images(pending_images)
                # 저장한 뒤에는 대기열을 비웁니다.
                pending_images = {label: [] for label in SAVE_DIRS}
                print(f"Saved {saved_count} face crop images.")
                continue

            if key == ord("q"):
                print("[q] quit.")
                break

    finally:
        camera_capture.release()
        cv2.destroyAllWindows()

[q] quit.


## 9. 수집 개수 확인

클래스별 저장 개수와 metadata 파일 위치를 확인합니다.

In [18]:
for label, save_dir in SAVE_DIRS.items():
    file_count = sum(1 for path in save_dir.glob("*.jpg"))
    print(f"{label}: {file_count}")

print(f"metadata: {METADATA_PATH}")

Attractive: 0
Drowsy: 0
Inattentive: 0
metadata: captured_images\metadata.csv
